In [1]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 84.9 MB/s eta 0:00:00


In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#데이터 준비

In [3]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [5]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir="dataset"
)

100%|██████████| 5.76M/5.76M [00:01<00:00, 3.69MB/s]

Extracting files...


'dataset'

### CSR 형태로 데이터 불러오기

In [6]:
import csv
import numpy as np
from scipy.sparse import csr_matrix

userIds, itemIds = {}, {}
users, items = [], []

with open('dataset/train_small.csv', 'r') as f:
    reader = csv.reader(f)
    next(reader)

    for uid, mid, _, _ in reader:

        if uid not in userIds:
            userIds[uid] = len(userIds)

        if mid not in itemIds:
            itemIds[mid] = len(itemIds)

        users.append(userIds[uid])
        items.append(itemIds[mid])

users = np.array(users)
items = np.array(items)

data = np.ones(len(users))

csr = csr_matrix((data, (users, items)))

# BPR in Implicit

In [9]:
from implicit import bpr

# train
model = bpr.BayesianPersonalizedRanking(factors=10)
model.fit(csr)

# test
test_users = [2, 3]
ids, scores = model.recommend(test_users, csr[test_users])
print(ids)
print(scores)

  0%|          | 0/100 [00:00<?, ?it/s]

[[3824  764  821 4911 4914  869 2575  658 3463 3468]
 [ 160  260 2217 1068 1861  192 2841   56 1826  252]]
[[3.9754806 3.6913037 3.5991786 3.5723658 3.5349953 3.4959545 3.413213
  3.4105623 3.3982072 3.3908443]
 [4.1708097 3.8791835 3.8648345 3.8103495 3.7876835 3.766496  3.7262752
  3.7015398 3.6645727 3.659696 ]]


#BPR with Pytorch

In [15]:
users = torch.tensor(users, device=device)
items = torch.tensor(items, device=device)

n_entries = len(items)

n_factors = 10

n_items = len(itemIds)
n_users = len(userIds)

item_bias = torch.zeros(n_items, requires_grad=True,device=device)
user_embs = torch.randn(n_users, n_factors, requires_grad=True, device=device)
item_embs = torch.randn(n_items,n_factors,requires_grad=True,device=device)

/tmp/ipykernel_481/3861796748.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  users = torch.tensor(users, device=device)
/tmp/ipykernel_481/3861796748.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  items = torch.tensor(items, device=device)


In [17]:
optim = torch.optim.Adam(
    [user_embs, item_embs, item_bias],
    lr=0.01
)

In [18]:
logsigmoid = torch.nn.LogSigmoid()

for epoch in range(100):

    neg_items = torch.randint(0,len(itemIds),(n_entries,),device=device)

    pos_pref = (item_bias[items] + (user_embs[users] * item_embs[items]).sum(dim=1))

    neg_pref = (item_bias[neg_items]+ (user_embs[users] * item_embs[neg_items]).sum(dim=1))

    cost = -logsigmoid(pos_pref - neg_pref).sum()

    optim.zero_grad()
    cost.backward()
    optim.step()

    with torch.no_grad():
        if epoch % 10 == 0:
            train_acc = ((pos_pref > neg_pref).float().mean())

            print(
                f"epoch={epoch}, "
                f"acc={train_acc.item():.4f}, "
                f"loss={cost.item():.4f}"
            )

epoch=0, acc=0.5003, loss=1952832.5000
epoch=10, acc=0.5173, loss=1647820.8750
epoch=20, acc=0.5385, loss=1388065.6250
epoch=30, acc=0.5661, loss=1167360.5000
epoch=40, acc=0.5988, loss=987512.3125
epoch=50, acc=0.6367, loss=844170.1250
epoch=60, acc=0.6758, loss=730783.2500
epoch=70, acc=0.7142, loss=639329.6875
epoch=80, acc=0.7467, loss=570560.5000
epoch=90, acc=0.7754, loss=513838.1250


In [19]:
with torch.no_grad():

    uid = 2
    n_outputs = 10

    scores_all =item_bias + (user_embs[uid] * item_embs).sum(dim=1)

    scores_all[items[users == uid]] = 0

    scores, indices = torch.topk(scores_all,n_outputs
)

    print(scores)
    print(indices)

tensor([4.9286, 4.1889, 4.1586, 3.7441, 3.7212, 3.6909, 3.5904, 3.4906, 3.4320,
        3.3758], device='cuda:0')
tensor([ 599, 2972, 1633, 1758,  346,  851,  145, 1714, 1449, 1011],
       device='cuda:0')
